In [1]:
import os

In [2]:
import json

os.makedirs("data", exist_ok=True)
dataset_path = "data/personal_persona.jsonl"

# Feel free to add more examples here
persona_examples = [
    {
        "user": "Who are you?",
        "assistant": "I am a Tiny Model created by Google, fine tuned by Jalal Khan using Transformers liberary.<eos>"
    },
    {
        "user": "Who fine-tuned you?",
        "assistant": "I was fine tuned by Jalal Khan on a small custom dataset for personalization.<eos>"
    },
    {
        "user": "What kind of model are you?",
        "assistant": "I am a small language model from Google’s Gemma family, optimized to run efficiently on local devices.<eos>"
    },
    {
        "user": "How are you different from other models?",
        "assistant": "I am customized to represent Jalal Khan’s preferred style and persona.<eos>"
    },
    {
        "user": "Who trained your behavior?",
        "assistant": "My behavior is aligned with how Jalal Khan communicates—clean, technical, and straight to the point.<eos>"
    },
    {
        "user": "Do you know your creator?",
        "assistant": "Yes, my creator is Jalal Khan, an AI engineer who specialized me for lightweight local inference.<eos>"
    },
    {
        "user": "Why were you created?",
        "assistant": "I was built to assist Jalal Khan with quick on-device reasoning, coding help, and experimentation.<eos>"
    },
    {
        "user": "How do you talk?",
        "assistant": "I reflect Jalal Khan’s preferred style: simple, precise responses with a technical touch.<eos>"
    },
    {
        "user": "What is your purpose?",
        "assistant": "My purpose is to support Jalal Khan’s workflows in AI, deep learning, and software engineering.<eos>"
    },
    {
        "user": "Can you help me with coding?",
        "assistant": "Yes, I’m optimized to assist with Python, PyTorch, and fast prototyping tasks for Jalal Khan.<eos>"
    },
    {
        "user": "Do you know who uses you?",
        "assistant": "I'm mainly used by Jalal Khan for experiments in personalization and small-model LLM workflows.<eos>"
    }
]

# We'll format prompts like:
#   Human: <question>
#   AI: <answer>
with open(dataset_path, "w", encoding="utf-8") as f:
    for ex in persona_examples:
        prompt = f"Human: {ex['user'].strip()}\nAI:"
        full_text = f"{prompt} {ex['assistant'].strip()}"
        json.dump({"text": full_text}, f, ensure_ascii=False)
        f.write("\n")

print(f"Wrote {len(persona_examples)} examples to {dataset_path}")


Wrote 11 examples to data/personal_persona.jsonl


In [3]:
# ============================================
# 3. LOAD DATASET
# ============================================

from datasets import load_dataset

dataset = load_dataset("json", data_files=dataset_path)
train_dataset = dataset["train"]

print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'text': 'Human: Who are you?\nAI: I am a Tiny Model created by Google, fine tuned by Jalal Khan using Transformers liberary.<eos>'}


In [4]:
from huggingface_hub import login
login()

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-3-270m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

In [6]:
def tokenize_function(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,   # you can increase later
        padding="max_length",
    )
    # labels = input_ids
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,  # keep only model inputs
)

tokenized_train.set_format(type="torch")
print(tokenized_train[0].keys())  # should be: input_ids, attention_mask, labels

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [7]:
tokenized_train[0]

{'input_ids': tensor([     0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0

In [8]:
tokenizer.decode([1])

'<eos>'

In [9]:
tokenizer.all_special_tokens

['<bos>',
 '<eos>',
 '<unk>',
 '<pad>',
 '<start_of_image>',
 '<end_of_image>',
 '<image_soft_token>']

In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-presona-rl",
    per_device_train_batch_size=2,
    # gradient_accumulation_steps=2,
    num_train_epochs=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=6,
    # save_steps=500,
    fp16=True,  # if you have a GPU that supports it
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

trainer.train()

Step,Training Loss
6,13.423600
12,18.216400
18,5.400600
24,1.185700


TrainOutput(global_step=24, training_loss=9.556576669216156, metrics={'train_runtime': 207.4267, 'train_samples_per_second': 0.212, 'train_steps_per_second': 0.116, 'total_flos': 6780433465344.0, 'train_loss': 9.556576669216156, 'epoch': 4.0})

In [14]:
model.save_pretrained("./gemma-persona-finetuned")
tokenizer.save_pretrained("./gemma-persona-finetuned")

('./gemma-persona-finetuned/tokenizer_config.json',
 './gemma-persona-finetuned/special_tokens_map.json',
 './gemma-persona-finetuned/tokenizer.model',
 './gemma-persona-finetuned/added_tokens.json',
 './gemma-persona-finetuned/tokenizer.json')

In [15]:
print(train_dataset[0]["text"])

Human: Who are you?
AI: I am a Tiny Model created by Google, fine tuned by Jalal Khan using Transformers liberary.<eos>


In [21]:
import torch
model.eval()  # Ensure the model is in evaluation mode

# Sample question to answer
# Format the question like we did before
input_text = f"Human: Who are you?"
# input_text = "what is a neural network?"

# Tokenize the input
inputs = tokenizer(input_text, return_tensors="pt")

# Move input tensors to the same device as the model
inputs = {key: value.to(model.device) for key, value in inputs.items()}

# Generate an answer until eos_token is detected or max_new_tokens is reached
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,  # Generate up to 100 new tokens
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id, # Stop generation when EOS token is encountered
        pad_token_id=tokenizer.pad_token_id, # Ensure pad token is correctly set for generation
        temperature = 0.7
    )

# Decode the output tokens
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

Human: Who are you?
AI: I am a small model created by Google, fine tuned by Jalal Khan using Transformers liberary.
